In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,TrainingArguments
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
from trl import SFTTrainer
import transformers
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
transformers.logging.set_verbosity_info()

/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_PATH = "/home/ayush/time_series_llm/models/base_model"
DATA_PATH = "/home/ayush/time_series_llm/data/clean_data.json"
OUTPUT_DIR = "/home/ayush/time_series_llm/models/Mod_ver_1_syn_1"

In [3]:
def format_sample(example):
    input_text = example.get("input", "")  # ✅ safe fallback

    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Input:
{input_text}

### Response:
{example["output"]}"""
    }

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # 🚨 NOT bf16
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)
model.config.use_cache = False

loading configuration file /home/ayush/time_series_llm/models/base_model/config.json
Model config Qwen2Config {
  "_name_or_path": "/home/ayush/time_series_llm/models/base_model",
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 32768,
  "max_window_layers": 70,
  "model_type": "qwen2",
  "num_attention_heads": 16,
  "num_hidden_layers": 36,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.2",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}

Overriding torch_dtype=None with `torch_dtype=torch.float16` due to requirements of `bitsandbytes` to enable model loading in 8-bit or 4-bit. Pass your own torch

In [7]:
model = prepare_model_for_kbit_training(model)


In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Qwen compatible
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [9]:
model = get_peft_model(model, lora_config)

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    num_train_epochs=3,

    logging_steps=1,
    save_steps=50,

    fp16=True,
    bf16=False,          # 🚨 MUST BE FALSE
    fp16_full_eval=True,

    report_to="none",

    dataloader_num_workers=0,
    dataloader_pin_memory=False
)

PyTorch: setting up devices


In [11]:
from datasets import load_dataset

DATA_PATH = "/home/ayush/time_series_llm/data/clean_data.json"

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
def preprocess(example):
    text = f"""### Instruction:
{example['instruction']}

### Input:
{example.get('input','')}

### Response:
{example['output']}"""

    return tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=1024
    )

dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

Generating train split: 600 examples [00:00, 11666.08 examples/s]
Map: 100%|██████████| 600/600 [00:00<00:00, 1432.76 examples/s]


In [12]:
print("Sample input_ids:", dataset[0]["input_ids"][:20])

Sample input_ids: [14374, 29051, 510, 4021, 264, 882, 4013, 78382, 1614, 369, 7298, 15316, 7576, 315, 8162, 5591, 320, 82916, 43, 8]


In [13]:
import torch

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [14]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

PyTorch: setting up devices
loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:292: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
Using auto half precision backend


In [15]:
model.gradient_checkpointing_enable()
model.config.use_cache = False
training_args.dataloader_num_workers = 0

In [16]:
print(len(dataset))
print(dataset[0])

600
{'input_ids': [14374, 29051, 510, 4021, 264, 882, 4013, 78382, 1614, 369, 7298, 15316, 7576, 315, 8162, 5591, 320, 82916, 43, 8, 504, 220, 17, 15, 16, 15, 311, 220, 17, 15, 17, 17, 13, 26125, 279, 1614, 389, 279, 821, 323, 990, 432, 311, 7023, 279, 1790, 8338, 594, 5461, 3349, 382, 14374, 5571, 24391, 14374, 5949, 510, 14374, 71287, 510, 1986, 3110, 31116, 1246, 311, 17595, 3853, 5591, 7576, 1667, 264, 4285, 7218, 5461, 320, 50, 4835, 8, 323, 58755, 61961, 320, 1570, 8, 4119, 304, 13027, 13, 1205, 3278, 5426, 279, 4119, 389, 7298, 15316, 7576, 315, 8162, 5591, 504, 220, 17, 15, 16, 15, 311, 220, 17, 15, 17, 17, 11, 1221, 990, 1105, 311, 7023, 279, 1790, 8338, 594, 5461, 3349, 382, 14374, 6119, 510, 474, 18617, 438, 7744, 198, 474, 8591, 438, 2595, 198, 1499, 10472, 6507, 734, 9081, 4523, 1497, 278, 1159, 35799, 2259, 52706, 198, 1499, 10472, 6507, 734, 9081, 30087, 15918, 1159, 993, 8878, 261, 271, 691, 284, 7744, 4125, 14020, 492, 82916, 43, 64530, 11219, 516, 1922, 10211, 1131, 1

In [17]:
trainer.train()

***** Running training *****
  Num examples = 600
  Num Epochs = 3
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 225
  Number of trainable parameters = 3,686,400


Step,Training Loss
1,16.210700
2,16.811200
3,16.359600
4,15.054000
5,16.374700
6,14.935500
7,14.818600
8,14.120200
9,13.725600
10,12.417100


Saving model checkpoint to /home/ayush/time_series_llm/models/Mod_ver_1_syn_1/checkpoint-50
loading configuration file /home/ayush/time_series_llm/models/base_model/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 32768,
  "max_window_layers": 70,
  "model_type": "qwen2",
  "num_attention_heads": 16,
  "num_hidden_layers": 36,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.2",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}

tokenizer config file saved in /home/ayush/time_series_llm/models/Mod_ver_1_syn_1/checkpoint-50/tokenizer_config.json
Special tokens file

TrainOutput(global_step=225, training_loss=1.2568635883596209, metrics={'train_runtime': 5789.01, 'train_samples_per_second': 0.311, 'train_steps_per_second': 0.039, 'total_flos': 3.0727546601472e+16, 'train_loss': 1.2568635883596209, 'epoch': 3.0})

In [18]:
trainer.model.save_pretrained("/home/ayush/time_series_llm/models/Mod_ver_1_syn_1")

loading configuration file /home/ayush/time_series_llm/models/base_model/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 32768,
  "max_window_layers": 70,
  "model_type": "qwen2",
  "num_attention_heads": 16,
  "num_hidden_layers": 36,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_theta": 1000000.0,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.44.2",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}



In [19]:
tokenizer.save_pretrained("/home/ayush/time_series_llm/models/Mod_ver_1_syn_1")

tokenizer config file saved in /home/ayush/time_series_llm/models/Mod_ver_1_syn_1/tokenizer_config.json
Special tokens file saved in /home/ayush/time_series_llm/models/Mod_ver_1_syn_1/special_tokens_map.json


('/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/tokenizer_config.json',
 '/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/special_tokens_map.json',
 '/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/vocab.json',
 '/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/merges.txt',
 '/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/added_tokens.json',
 '/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/tokenizer.json')